## Object Detection in MS COCO

# Full Image-Set Test Clone

This clone is stripped of outputs and has a corrected full-pipeline cell for running all partitioned images.


In [ ]:
import os
import torch
import numpy as np
from torchvision import transforms
#--------------------------------------------
from skimage.filters import gaussian
from scipy.ndimage import gaussian_filter
import cv2

import matplotlib.pyplot as plt

import shap_bpt
print('shap_bpt version:',shap_bpt.__version__)
print('shap_bpt release name:',shap_bpt.__release_name__)


## Set Config

In [ ]:
import os
from pathlib import Path
import yaml

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'shap_bpt').is_dir():
            return path
    raise FileNotFoundError('Could not find the project root from the current working directory.')

original_working_dir = Path.cwd()
project_root = find_project_root(original_working_dir)
os.chdir(project_root)
print(f'Project root: {project_root}')

config_file = "MSCOCO_mac"
# config_file = "MSCOCO_xn2"

try:
    with open(project_root / f"examples/configs/{config_file}.yaml", "r") as f:
        config = yaml.safe_load(f)
finally:
    os.chdir(original_working_dir)
    print(f'Restored working directory: {original_working_dir}')

dataset_root = config["data"]["dataset_root"]
print(f"Dataset root: {dataset_root}")

print(f"Masks Path  : {config['data']['masks_path']}")

masks_base_path = os.path.join(config['data']['masks_path'], config['data']['mask_dir'], config['data']['mask_dir_final'])

print(f"masks_base_path Path  : {masks_base_path}")

In [ ]:
import sys

scripts_dir = project_root / "examples/scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import utils_xai as utx
import utils_sam as uts

import importlib
importlib.reload(utx) 
importlib.reload(uts)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if ('mps' in dir(torch.backends)) and torch.backends.mps.is_available() else torch.device("cpu")
device

In [ ]:
config_compiled={}
config_compiled['device'] = device


In [ ]:
from ultralytics import YOLO

model = YOLO(f'{original_working_dir}/checkpoints/yolo11s.pt')
class_names = model.names

print(f'{"Num_Classes":<15}{len(class_names)}')

model.info()

from pycocotools.coco import COCO

model_preprocess = transforms.Compose(
    [transforms.ToTensor()]
)

In [ ]:
# def load_image(fname,im_size=None,bg_type='black'):
#     img_ = cv2.imread(f'{fname}')
#     image_to_explain = cv2.cvtColor(img_, cv2.COLOR_BGR2RGB)                #.astype(np.float32)
#     if im_size is not None:
#         image_to_explain         = cv2.resize(image_to_explain,im_size)     # [:,:,::-1]
#     image_to_explain_preproc  = image_to_explain.copy()                     #torch.tensor(image_to_explain).to(device)# .astype(np.float32)/255.0
#     np.random.seed(0)
#     bkgnd0 = np.full_like(image_to_explain, 0)
#     bkgnd1 = np.full_like(image_to_explain, 127)
#     bkgnd2 = np.full_like(image_to_explain, 255)
#     bkgnd3 = gaussian(image_to_explain, 8, channel_axis=-1)*255
#     bkgnd4 = np.clip(np.random.normal(128, 128, size=image_to_explain.shape), 0, 255).astype(np.uint8)
#     bkgnd4 = (gaussian(bkgnd4, 2.0, channel_axis=-1) * 255).astype(np.uint8)
#     if bg_type=='black': background_image_set = np.array([bkgnd0])
#     elif bg_type=='gray': background_image_set = np.array([bkgnd1])
#     elif bg_type=='white': background_image_set = np.array([bkgnd2])
#     elif bg_type=='blurred': background_image_set = np.array([bkgnd3])
#     elif bg_type=='noise': background_image_set = np.array([bkgnd4])
#     elif bg_type=='full': background_image_set = np.array([bkgnd0, bkgnd1, bkgnd2, bkgnd3, bkgnd4])
#     else: raise ValueError(f'Unknown bg_type: {bg_type}')
#     background_image_preproc_set = [model_preprocess(bkgnd.astype(np.float32)/255.0)
#                                         for bkgnd in background_image_set]
#     background_tensors = torch.cat([torch.unsqueeze(bk_p, dim=0) 
#                                     for bk_p in background_image_preproc_set]).to(device)
#     return image_to_explain,image_to_explain_preproc,background_image_set,background_tensors

In [ ]:
# def predict_yolo(x,coco_classes_count=80,verbose=False):
#     res = model.predict(source=x, verbose=verbose)[0]
#     p = np.zeros(coco_classes_count)
#     for cls,prob in zip(res.boxes.cls.cpu().numpy(), res.boxes.conf.cpu().numpy()):
#         p[int(cls)] = prob
#     return np.array(p)
# #-----------------------------------------------------------------------
# def predict_yolo_masked(masks,verbose=False):
#     imglst_preds = []
#     for mask in masks:
#         preds = []
#         for repl in background_image_set:
#             # print(mask.shape, repl.shape)
#             if len(mask.shape)!=3:
#                 mask3 = np.stack([mask,mask,mask], axis=2)
#             else:
#                 print(mask.shape)
#                 mask3 = mask.copy()
#             masked_image = np.where(mask3, image_to_explain, repl)
#             preds.append(predict_yolo(masked_image,verbose=verbose))

#         preds = np.mean(preds, axis=0)
#         imglst_preds.append(preds)       
    
#     return np.array(imglst_preds)

In [ ]:
def load_image_to_explain(fname,bg_type='gray', load_gt=True):
    # global predicted_fS, predicted_f0, predicted_cls, sorted_classes, f_S, f_0,sorted_probs
    # global model_type,pretrained_model_type

    image_to_explain,image_to_explain_tensor,background_image_set,background_tensors = utx.load_image(fname,bg_type=bg_type)
    h,w,_ = image_to_explain.shape
    # Foreground image to be explained  
    predicted_fS = utx.predict_yolo(model,image_to_explain) 
    # predicted_fS = f(torch.unsqueeze(resnet50_preprocess(image_to_explain.astype(np.float32)/255.0).to(device), dim=0))[0]
    sorted_classes = np.flip(np.argsort(predicted_fS))
    sorted_probs   = predicted_fS[sorted_classes]
    predicted_cls = sorted_classes[0]
    f_S = float(predicted_fS[predicted_cls])
    #####################
    
    predicted_f0 = [utx.predict_yolo(model, bkgnd.astype(np.float32)/255.0) for bkgnd in background_image_set]
    predicted_f0 = np.mean(predicted_f0,axis=0)
    f_0          = float(predicted_f0[predicted_cls])
    return image_to_explain,image_to_explain_tensor,background_image_set,background_tensors,predicted_fS,sorted_classes,sorted_probs,predicted_cls,f_S,predicted_f0,f_0
    # if load_gt:
        # image_no = int(image_path.split('\\')[-1].split('.')[0])

        # load_groundtruth(coco,image_path,fixed_category=fixed_category)
    

## Load Partitions

In [ ]:
path_partition = f'../partitions_100/'


## SELECT IMAGE

In [ ]:
## Get available precomputed SAM partitions
# fetch available unique image_ids from the partitions directory
image_ids = os.listdir(path_partition)
image_ids = [f.split('.')[0].split('_')[0] for f in image_ids if '_refined.npy' in f]
print(f'computed image_ids: {len(image_ids)}')

already_computed = ['000000049091',
 '000000000632',
 '000000186929',
 '000000002299',
 '000000225757',
 '000000171382']

image_ids = [img_id for img_id in image_ids if img_id not in already_computed]
print(f'filtered image_ids: {len(image_ids)}')
image_ids[0:5]

In [ ]:
image_dir = config["data"]["image_dir"] # Update for your image directory
annotation_file =  config["data"]["annotation_file"] # Update for your annotation file
coco = COCO(annotation_file)

categories = coco.loadCats(coco.getCatIds())
coco_categories = {cat['id']: cat['name'] for cat in categories}

In [ ]:
# image_id = '000000171382'  # Example image ID
# image_id = '000000170670'  # Example image ID
image_id = image_ids[0]  # Example image ID
image_path = os.path.join(image_dir, f'{image_id}.jpg')
print(f'Image path: {image_path}')

In [ ]:
image_to_explain,image_to_explain_tensor,background_image_set,background_tensors,\
    predicted_fS,sorted_classes,sorted_probs,predicted_cls,f_S,predicted_f0,f_0 = load_image_to_explain(image_path, bg_type='gray')
# fixed_category = 'tv'
fixed_category = class_names[predicted_cls]
print('fixed_category: ',fixed_category)

In [ ]:
plt.imshow(image_to_explain)
plt.xticks([]); plt.yticks([]); 
plt.title(f'Predicted Class: {fixed_category}'); plt.show()


In [ ]:
# image_no = 113235
image_no = int(image_id)

image_info = coco.loadImgs(image_no)[0]
ann_ids = coco.getAnnIds(imgIds=image_info['id'])
annotations = coco.loadAnns(ann_ids)
has_segmentation = any('segmentation' in ann for ann in annotations)
print(f"Segmentation annotations present: {has_segmentation}")

In [ ]:
# image_info

In [ ]:
# categories = coco.loadCats(coco.getCatIds())
# coco_categories = {cat['id']: cat['name'] for cat in categories}
# Get annotations for the selected image
ann_ids = coco.getAnnIds(imgIds=image_info['id'])
annotations = coco.loadAnns(ann_ids)

In [ ]:
importlib.reload(utx)
utx.check_annotation(annotations)

In [ ]:
importlib.reload(utx)   
ground_truth,weighted_ground_truth=utx.load_groundtruth(coco,image_no,image_to_explain,image_info,fixed_category=fixed_category)

In [ ]:
plt.imshow(ground_truth, cmap='gray'); plt.xticks([]); plt.yticks([]); plt.title(f'Ground Truth Mask for {fixed_category}'); plt.show()

In [ ]:
# # Perform inference on an image
# def plot_predictions(image, results, category_name=None, filter_preds=True,
#                      line_thickness=2, exp_type='demo', save_fig=False, fig_size=(3, 3),
#                      title=None, selected_ext='png', destroy_fig=False):
#     image_ = image.copy()
#     plt.figure(figsize=fig_size)
#     plt.imshow(image_); plt.axis('off')
#     for result in results:
#         # Access detected classes, confidences, and boxes
#         class_ids = result.boxes.cls.cpu().numpy()  # Class IDs
#         scores = result.boxes.conf.cpu().numpy()   # Confidence scores
#         boxes = result.boxes.xyxy.cpu().numpy()    # Bounding boxes in xyxy format
#         labels = model.names                       # Class labels (MS COCO classes)
#         # Draw bounding boxes and labels on the image
#         for box, class_id, score in zip(boxes, class_ids, scores):
#             label = labels[int(class_id)]
#             confidence = f"{score:.2f}"
#             x1, y1, x2, y2 = map(int, box)  # Bounding box coordinates
#             if filter_preds and category_name and label != category_name:
#                 continue
#             width,height = x2 - x1, y2 - y1
#             plt.gca().add_patch(plt.Rectangle((x1, y1), width, height, edgecolor='darkred', facecolor='none', linewidth=line_thickness))
#             plt.text(x1, y1 - 5, f"{label} {confidence}", color='white', fontsize=12, bbox=dict(facecolor='darkred', alpha=0.5))
#     plt.show()


In [ ]:
importlib.reload(utx)
results = model.predict(image_to_explain,verbose=True)

In [ ]:
top_classes_n = 5

top_k_classes = utx.get_top_k_classes(results,model.names, k=5)
print("Top-k Classes (Class ID, Confidence):")
print(top_k_classes)

# print([class_names[int(cls)] for cls, _,_ in top_k_classes])
# print('-'*120)
# print('top_k_classes \t', top_k_classes)

In [ ]:
fixed_category

In [ ]:
utx.plot_predictions(image_to_explain,model,results,
                     category_name = fixed_category,
                     fig_size=(5,5),
                     save_fig=False)

In [ ]:
# MASKING FUNCTION
def predict_yolo_masked(masks):
    imglst_preds = []
    for mask in masks:
        preds = []
        for repl in background_image_set:
            if len(mask.shape)==2:
                mask3 = np.stack([mask,mask,mask], axis=2)
            else:
                mask3 = mask.copy()
            masked_image = np.where(mask3, image_to_explain, repl)
            preds.append(predict_yolo(masked_image))
        preds = np.mean(preds, axis=0)
        imglst_preds.append(preds)       
    return np.array(imglst_preds)

def predict_yolo(x, coco_classes_count=80,verbose=False):
    res = model.predict(source=x, verbose=verbose)[0]
    p = np.zeros(coco_classes_count)

    for cls, prob in zip(res.boxes.cls.cpu().numpy(), res.boxes.conf.cpu().numpy()):
        cls = int(cls)
        p[cls] = max(p[cls], float(prob))

    return p

In [ ]:
importlib.reload(utx)
results = model.predict(image_to_explain, verbose=True)
top_k_classes = utx.get_top_k_classes(results, model.names, k=5)

predicted_fS = utx.predict_yolo(model,image_to_explain)
predicted_cls = int(np.argmax(predicted_fS))

print("YOLO top box:", top_k_classes[0])
print("predict_yolo top class:", predicted_cls, model.names[predicted_cls], predicted_fS[predicted_cls])

In [ ]:
print(image_to_explain.shape)
bg_ls = np.zeros((100,image_to_explain.shape[0],image_to_explain.shape[1],image_to_explain.shape[2]))
# bg_ls = np.random.randint(0, 255, (50,426, 640, 3), dtype=np.uint8)
print(bg_ls.shape)
pred = predict_yolo_masked(bg_ls)
print(pred.shape)
torch.cuda.empty_cache()

In [ ]:
num_explained_classes        = 1
MAX_EVALS_BUDGET             = 500
explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)

In [ ]:
shap_values = {}

In [ ]:
shap_values['BPT'] = explainer.explain_instance(MAX_EVALS_BUDGET, method='BPT',batch_size=16)
# shap_values['AA'] = explainer.explain_instance(MAX_EVALS_BUDGET, method='AA',  batch_size=16)

In [ ]:
shap_bpt.plot_owen_values(explainer, [shap_values['BPT']],class_names, names=['BPT'])

In [ ]:
path_results = os.path.join(original_working_dir, 'results')
path_results_img = os.path.join(path_results, str(image_no))
os.makedirs(path_results_img, exist_ok=True)

In [ ]:
# Save all detection results for each image to a JSON file used by the HTML report.
import json

def to_jsonable(value):
    if hasattr(value, 'item'):
        return value.item()
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    return value

if isinstance(results, dict):
    yolo_speed = results.get('speed', {})
elif isinstance(results, (list, tuple)) and len(results) > 0:
    yolo_speed = getattr(results[0], 'speed', {})
else:
    yolo_speed = getattr(results, 'speed', {})

detection_summary = {
    'image_id': str(image_no),
    'image_id_padded': str(image_id),
    'input_image_path': image_path,
    'sam_mask_path': str(project_root / 'examples' / 'partitions' / f'{image_id}_sam.png'),
    'fixed_category': fixed_category,
    'explained_class': fixed_category,
    'predicted_class_id': int(predicted_cls),
    'f_S': float(f_S),
    'f_0': float(f_0),
    'has_segmentation': bool(has_segmentation),
    'speed': to_jsonable(yolo_speed),
    'top_k_classes': [
        {'class_id': int(class_id), 'class_name': str(class_name), 'confidence': float(confidence)}
        for class_id, class_name, confidence in top_k_classes
    ],
}

detection_summary_path = os.path.join(path_results_img, f'{image_no}_results.json')
with open(detection_summary_path, 'w') as f:
    json.dump(to_jsonable(detection_summary), f, indent=2)

print(f'Saved detection summary: {detection_summary_path}')


## Distance Functions

### ShapBPT with Custom BPT

In [ ]:
print('Expected Shapley explanation: ', explainer.base_f_S[0] - explainer.base_f_0[0])
print('Computed Shapley explanation: ', np.sum(shap_values['BPT'][0]))

## Genearte/LOAD SAM

In [ ]:
partition = cv2.imread(f'{path_partition}/{image_id}_sam.png',
                    #    cv2.IMREAD_GRAYSCALE
                       )
plt.imshow(partition, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
masks_sorted = np.load(f"{path_partition}/{image_id}_sorted.npy").astype(np.uint16)
masks_refined = np.load(f"{path_partition}/{image_id}_refined.npy").astype(np.uint16)

In [ ]:
importlib.reload(uts)
uts.plot_masks([masks_sorted,masks_refined])

### ONLY SAM

In [ ]:
path_results = os.path.join(original_working_dir, 'results')
path_results_img = os.path.join(path_results, str(image_no))
os.makedirs(path_results, exist_ok=True)
os.makedirs(path_results_img, exist_ok=True)


## CREATE A MAPPING TABLE for this


In [ ]:
import pandas as pd

# Full-pipeline tests use the same methods as the current single-image notebook.
# No area/perimeter/color grid is swept here.
experiment_map = {
    "BPT": {
        "description": "BPT",
        "use_area_term": True,
        "use_perim_term": True,
        "use_color_term": True,
        "use_sam_partitions": False,
    },
    "SAM_S": {
        "description": "SAM sorted partitions only",
        "use_area_term": False,
        "use_perim_term": False,
        "use_color_term": False,
        "use_sam_partitions": True,
        "partition_source": "sorted",
    },
    # "SAM_R": {
    #     "description": "SAM refined partitions only",
    #     "use_area_term": False,
    #     "use_perim_term": False,
    #     "use_color_term": False,
    #     "use_sam_partitions": True,
    #     "partition_source": "refined",
    # },
    "BPT_SAM_S": {
        "description": "BPT + sorted SAM partitions",
        "use_area_term": True,
        "use_perim_term": True,
        "use_color_term": True,
        "use_sam_partitions": True,
        "partition_source": "sorted",
    },
    "BPT_SAM_R": {
        "description": "BPT + refined SAM partitions",
        "use_area_term": True,
        "use_perim_term": True,
        "use_color_term": True,
        "use_sam_partitions": True,
        "partition_source": "refined",
    },
}

experiment_table = pd.DataFrame.from_dict(experiment_map, orient="index")
experiment_table.index.name = "Exp"
experiment_table = experiment_table.reset_index()

print(experiment_table[["Exp", "description", "use_area_term", "use_perim_term", "use_color_term", "use_sam_partitions"]])


def get_experiment_config(code: str):
    return experiment_map[code]


def explain_experiment(code: str):
    cfg = get_experiment_config(code)
    return f"{code}: {cfg.get('description', code)}"


In [ ]:
# print(get_experiment_code(True, True, True))   # E1
# print(get_experiment_config("E1"))
# print(explain_experiment("E1"))
# print(explain_flags(True, True, True))

In [ ]:
MAX_EVALS_BUDGET = 500

In [ ]:
# print(partitions.dtype, partitions.shape)
# print("min/max:", partitions.min(), partitions.max())
# print("negative pixels:", np.sum(partitions < 0))
# print("labels:", np.unique(partitions)[:20])

### ONLY SAM (Sorted)

In [ ]:
explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)

partitions = uts.cap_partition_labels(masks_sorted, max_labels=63)
partitions_sorted = uts.sanitize_partitions(partitions, image_to_explain.shape, max_labels=63)

bptree = shap_bpt.build_bpt_from_image(
                        image_to_explain,
                        use_perim_term=False,
                        use_area_term=False,
                        use_color_term=False,
                        prebuilt_partitions=partitions_sorted,
)
print(bptree.N)
# Compute Owen values with the BPT method
shap_values['SAM_S'] = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)

In [ ]:
shap_bpt.plot_owen_values(explainer, shap_values['SAM_S'], class_names, names=['SAM_S'])

### ONLY SAM (Refined)

In [ ]:
partitions = uts.cap_partition_labels(masks_refined, max_labels=63)
partitions_refined = uts.sanitize_partitions(partitions, image_to_explain.shape, max_labels=63)

bptree = shap_bpt.build_bpt_from_image(
                        image_to_explain,
                        use_perim_term=False,
                        use_area_term=False,
                        use_color_term=False,
                        prebuilt_partitions=partitions_refined,
)
print(bptree.N)
# Compute Owen values with the BPT method
shap_values['SAM_R'] = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)

In [ ]:
shap_bpt.plot_owen_values(explainer, shap_values['SAM_R'], class_names, names=['SAM_R'])

## BPT + SAM(Sorted)

In [ ]:
explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)

partitions = uts.cap_partition_labels(masks_sorted, max_labels=63)
partitions_sorted = uts.sanitize_partitions(partitions, image_to_explain.shape, max_labels=63)

bptree = shap_bpt.build_bpt_from_image(image_to_explain,
                                       prebuilt_partitions=partitions_sorted
                                       )
print('Sorted BPTree N:', bptree.N)
# Compute Owen values with the BPT method
shap_values['BPT_SAM_S'] = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)
shap_bpt.plot_owen_values(explainer, shap_values['BPT_SAM_S'], class_names, names=['BPT+SAM_S'])

## BPT + SAM(Refined)

In [ ]:
partitions = uts.cap_partition_labels(masks_refined, max_labels=63)
partitions_refined = uts.sanitize_partitions(partitions, image_to_explain.shape, max_labels=63)

bptree = shap_bpt.build_bpt_from_image(image_to_explain,
                                        use_perim_term=True,
                                          use_area_term=True,
                                            use_color_term=True,
                                            prebuilt_partitions=partitions_refined
                                            )
print('Refined BPTree N:', bptree.N)
# Compute Owen values with the BPT method
shap_values['BPT_SAM_R'] = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =bptree, method='BPT',batch_size=16)
shap_bpt.plot_owen_values(explainer, shap_values['BPT_SAM_R'], class_names, names=['BPT+SAM_R'])

In [ ]:
shap_bpt.plot_owen_values(explainer, 
                          [shap_values['BPT'],
                           shap_values['SAM_S'],
                        #    shap_values['SAM_R'],
                           shap_values['BPT_SAM_S'],
                           shap_values['BPT_SAM_R']],
          class_names, names=['BPT','SAM_S','SAM_R','BPT+SAM_S','BPT+SAM+R'])

In [ ]:
importlib.reload(utx)
shapley_values_colormap = utx.get_shapley_values_colormap()
def plot_xai(shap_values):
    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    axes = axes.flatten()
    axes[0].imshow(image_to_explain)
    axes[0].set_title(f"Original Image - {image_id}")
    axes[0].axis('off')
    ## Plot yolo predictions
    utx.plot_predictions(image_to_explain, model, results,
                         category_name=fixed_category,
                         fig_size=(5, 5),
                         save_fig=False,
                         ax=axes[1])
    axes[1].set_title(f"YOLO Predictions for {fixed_category}")

    axes[2].imshow(partition, cmap='gray')
    axes[2].set_title(f"partition")
    axes[2].axis('off')

    for i, (method,sv) in enumerate(shap_values.items()):
        
        # axes[i + 1].imshow(shap_values[method][0], cmap='hot')
        sv,mv = utx.scale_shap_values(sv, robust_percentile=99.5, factor=1)
        max_abs = np.max(np.abs(sv))
        print(method, max_abs, mv)
        im = axes[i + 3].imshow(sv[0], vmin=-mv, vmax=mv, cmap=shapley_values_colormap)
        fig.colorbar(im, ax=axes[i + 3], fraction=0.03)#, location='bottom') #,  fraction=0.5, 
        axes[i + 3].set_title(f"{method} Explanation")
        # axes[i + 1].axis('off')
        axes[i + 3].set_xticks([]); axes[i + 3].set_yticks([])
    plt.tight_layout()
    # plt.subplots_adjust(wspace=0.1, hspace=0.2)
    plt.show()

In [ ]:
plot_xai(shap_values)

In [ ]:
# Disabled in the full image-set test clone.
# Original cell started with: for exp_code, sv in shap_values.items():
# Use the corrected full-pipeline cell below instead.


## ALL Distance Functions

In [ ]:
# # Create the shap_bpt explainer using our masking-based black-box function
# explainer   = shap_bpt.Explainer(predict_yolo_masked, image_to_explain, num_explained_classes=num_explained_classes, verbose=True)
# partitions = cap_partition_labels(masks, max_labels=63)
# from tqdm.auto import tqdm
# # add a progress bar for the nested loops based on all combinations of the four boolean flags (2^4 = 16 combinations)
# combination_count = 2 ** 4 # 2^4 combinations
# pbar = tqdm(total=combination_count)  
# shap_values_all = []

# for use_bpt in [True]:
#     for use_refined_partitions in [True, False]:
#         partitions = cap_partition_labels(masks_refined if use_refined_partitions else masks_sorted, max_labels=63) 
#             for use_sam_partitions_refined in [True, False]:
#                 bptree = shap_bpt.build_bpt_from_image(image_to_explain,
#                                                        prebuilt_partitions=partitions if use_sam_partitions else None
#                                                        )
#                 # Compute Owen values with the BPT method
#                 shap_values_bpt_sam = explainer.explain_instance(MAX_EVALS_BUDGET, method='BPT',
#                                                             bpt=bptree,
#                                                             batch_size=64)
#                 active_terms = []
#                 if use_perim_term: active_terms.append('Perimeter')
#                 if use_area_term: active_terms.append('Area')
#                 if use_color_term: active_terms.append('Color')
#                 if use_sam_partitions: active_terms.append('SAM')
#                 string_names = ' | '.join(active_terms) or 'No terms'

#                 exp_code = get_experiment_code(use_area_term, use_perim_term, use_color_term)
#                 figure_name = f'{exp_code}_{"SAM" if use_sam_partitions else "noSAM"}_{image_no}.png'

#                 plot_owen_values(explainer, shap_values_bpt_sam, class_names,figure_name, names=[string_names], save_plot=False, savepath= path_results_img)
#                 plot_single_attributions(shap_values_bpt_sam, path_results_img,  figure_name)
#                 data = {'exp_code':exp_code,
#                  'shap_values':shap_values_bpt_sam[0]
#                  }
#                 shap_values_all.append(data)
#                 pbar.update(1)  # Update the progress bar after each combination
#                 # plt.gcf().savefig(os.path.join(path_results, figure_name), dpi=300, bbox_inches='tight')
#                 # print(f'Saved figure: {figure_name}')
#                 ## save this plot if needed

## Generate Report for all Images

In [ ]:
# Disabled in the full image-set test clone.
# Original cell started with: utx.build_all_images_html_report(
# Use the corrected full-pipeline cell below instead.


## Evaluation

In [ ]:
def _iter_shap_value_items(shap_values):
    if isinstance(shap_values, dict):
        yield from shap_values.items()
        return

    for idx, exp_data in enumerate(shap_values):
        if isinstance(exp_data, dict):
            exp_code = exp_data.get('exp_code', exp_data.get('label', f'exp_{idx}'))
        else:
            exp_code = f'exp_{idx}'
        yield exp_code, exp_data


def _auc_heatmap(exp_data, auc_class_index=0):
    if isinstance(exp_data, dict):
        raw = exp_data.get('shap_values', exp_data.get('values', exp_data.get('heatmap')))
    else:
        raw = exp_data

    if raw is None:
        raise ValueError('Could not find shap values. Expected an array or a dict with shap_values/values/heatmap.')

    heatmap = np.asarray(raw)
    heatmap = np.squeeze(heatmap)

    if heatmap.ndim == 3:
        if auc_class_index >= heatmap.shape[0]:
            raise ValueError(f'auc_class_index={auc_class_index} is outside shap value stack with shape {heatmap.shape}')
        heatmap = heatmap[auc_class_index]

    if heatmap.ndim != 2:
        raise ValueError(f'Expected one 2D attribution map after squeezing/indexing, got shape {heatmap.shape}')

    return heatmap.astype(float, copy=False)


def compute_auc_results(shap_values, nu, f_S, f_0, predicted_cls, batch_size=4,
                        auc_class_index=0, verbose=True):
    auc_results = []

    for exp_code, exp_data in _iter_shap_value_items(shap_values):
        heatmap = _auc_heatmap(exp_data, auc_class_index=auc_class_index)
        auc_del = utx.saliency_to_auc(nu, heatmap, f_S, f_0, predicted_cls, batch_size=batch_size, method='del')
        auc_ins = utx.saliency_to_auc(nu, heatmap, f_S, f_0, predicted_cls, batch_size=batch_size, method='ins')

        result = {
            'label': str(exp_code),
            'auc_del': auc_del,
            'auc_ins': auc_ins,
        }
        if isinstance(exp_data, dict):
            result.update({
                k: v for k, v in exp_data.items()
                if k not in {'shap_values', 'values', 'heatmap'}
            })

        auc_results.append(result)
        if verbose:
            print(f"{exp_code}: AUC-DEL={auc_del['auc_clipr']:.4f}, AUC-INS={auc_ins['auc_clipr']:.4f}")

    return auc_results


def plot_auc_results(auc_results, figsize=(11, 4), fill_alpha=0.08, save_plot=False,
                     image_no=None,
                     destroy_figs=False):
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=True, sharey=True)
    cmap = plt.get_cmap('tab20')

    best_ins = max(r['auc_ins']['auc_clipr'] for r in auc_results)
    best_del = min(r['auc_del']['auc_clipr'] for r in auc_results)

    panels = [
        (axes[0], 'auc_ins', best_ins, True, '$\\mathit{AUC}^{+}$', 'lower right'),
        (axes[1], 'auc_del', best_del, False, '$\\mathit{AUC}^{-}$', 'upper right'),
    ]

    for ax, auc_key, best_auc, higher_is_better, title, legend_loc in panels:
        sorted_results = sorted(
            auc_results,
            key=lambda result: result[auc_key]['auc_clipr'],
            reverse=higher_is_better,
        )

        for idx, result in enumerate(sorted_results):
            auc = result[auc_key]
            score = auc['auc_clipr']
            is_best = np.isclose(score, best_auc)
            color = cmap(idx % cmap.N)
            star = '*' if is_best else ''

            ax.plot(
                auc['xs'],
                auc['y_clipr'],
                color=color,
                lw=2.4 if is_best else 1.4,
                alpha=0.95 if is_best else 0.75,
                label=f"{star}{result['label']} {score:.4f}",
            )
            ax.fill_between(auc['xs'], auc['y_clipr'], color=color, alpha=fill_alpha)

        ax.axhline(1.0, ls='--', c='grey', zorder=0)
        ax.axhline(0.0, c='lightgrey', zorder=0)
        ax.set_title(title, fontsize=16)
        ax.set_xlabel('Fraction of Pixels Removed/Inserted')
        ax.grid(alpha=0.25)
        ax.legend(borderpad=0.2, labelspacing=0.1, loc=legend_loc, fontsize=8)

    axes[0].set_ylabel('Model Confidence')
    plt.tight_layout()
    if save_plot:
        plt.savefig(os.path.join(path_results_img, f'auc_results_{image_no}.png'), dpi=300, bbox_inches='tight')
    if destroy_figs:
        plt.close(fig)
    plt.show()
    return fig, axes


In [ ]:
# Disabled in the full image-set test clone.
# Original cell started with: predicted_cls = explainer.output_indexes[0]
# Use the corrected full-pipeline cell below instead.


In [ ]:
# Disabled in the full image-set test clone.
# Original cell started with: for exp_code, sv in shap_values.items():
# Use the corrected full-pipeline cell below instead.


In [ ]:
# Disabled in the full image-set test clone.
# Original cell started with: # for exp_data in shap_values:
# Use the corrected full-pipeline cell below instead.


In [ ]:
# Disabled in the full image-set test clone.
# Original cell started with: shap_values.keys()
# Use the corrected full-pipeline cell below instead.


In [ ]:
# Disabled in the full image-set test clone.
# Original cell started with: auc_results = compute_auc_results(shap_values, predict_yolo_masked, f_S, f_0, predicted_cls, batch_size=4)
# Use the corrected full-pipeline cell below instead.


In [ ]:
# Disabled in the full image-set test clone.
# Original cell started with: report_path = utx.build_html_report(
# Use the corrected full-pipeline cell below instead.


In [ ]:
def auc_results_to_rows(auc_results, image_no=None, f_S=None, f_0=None,
                        image_id=None, fixed_category=None):
    rows = []
    for result in auc_results:
        rows.append({
            'image_no': int(image_no) if image_no is not None else None,
            'image_id': str(image_id) if image_id is not None else None,
            'fixed_category': fixed_category,
            'f_S': float(f_S) if f_S is not None else None,
            'f_0': float(f_0) if f_0 is not None else None,
            'method': result.get('label', result.get('exp_code', 'unknown')),
            'auc_ins': float(result['auc_ins']['auc_clipr']),
            'auc_del': float(result['auc_del']['auc_clipr']),
        })
    return rows


## RUN FULL PIPELINE

In [ ]:
# Full image-set run switch for this cloned test notebook.
run_fullpipeline = True


In [ ]:
path_partition_folder = 'partitions_100'  # or 'sorted'

In [ ]:
# Full image-set test run. This cell mirrors the current single-image method set:
# BPT, SAM_S, SAM_R, BPT_SAM_S, and BPT_SAM_R. No flag grid is swept.
save_plots = True
compute_auc = True
destroy_figs = True
verbose = False
skip_existing_reports = False

MAX_EVALS_BUDGET = 500
full_image_limit = None  # set to an int for a quick smoke test, e.g. 2

failures = []
completed_reports = []
auc_table_rows = []
path_results = os.path.join(original_working_dir, 'results')
os.makedirs(path_results, exist_ok=True)

if run_fullpipeline:
    import json
    import traceback
    from tqdm.auto import tqdm

    num_explained_classes = 4
    eval_batch_size = 16
    auc_batch_size = 4

    run_image_ids = list(image_ids)
    if full_image_limit is not None:
        run_image_ids = run_image_ids[:full_image_limit]

    for image_id_raw in tqdm(run_image_ids, desc='Full image-set XAI'):
        image_id = f"{int(image_id_raw):012d}"
        image_no = int(image_id)
        report_html = os.path.join(path_results, f'shapbpt_report_{image_no}.html')

        if skip_existing_reports and os.path.exists(report_html):
            if verbose:
                print(f'Skipping existing report: {report_html}')
            completed_reports.append(report_html)
            continue

        try:
            print('=' * 100)
            print(f'Image: {image_id}')

            image_dir = config['data']['image_dir']
            image_path = os.path.join(image_dir, f'{image_id}.jpg')
            sam_path = project_root / 'examples' / path_partition_folder / f'{image_id}_sam.png'
            sorted_path = project_root / 'examples' / path_partition_folder / f'{image_id}_sorted.npy'
            refined_path = project_root / 'examples' / path_partition_folder / f'{image_id}_refined.npy'

            if not os.path.exists(image_path):
                raise FileNotFoundError(f'Missing COCO image: {image_path}')
            if not sam_path.exists():
                raise FileNotFoundError(f'Missing SAM mask preview: {sam_path}')
            if not refined_path.exists():
                raise FileNotFoundError(f'Missing refined partitions: {refined_path}')

            image_to_explain, image_to_explain_tensor, background_image_set, background_tensors,                 predicted_fS, sorted_classes, sorted_probs, predicted_cls, f_S, predicted_f0, f_0 =                     load_image_to_explain(image_path, bg_type='gray')

            fixed_category = class_names[predicted_cls]
            print('fixed_category:', fixed_category)

            image_info = coco.loadImgs(image_no)[0]
            ann_ids = coco.getAnnIds(imgIds=image_info['id'])
            annotations = coco.loadAnns(ann_ids)
            has_segmentation = any('segmentation' in ann for ann in annotations)

            results = model.predict(image_to_explain, verbose=False)
            yolo_speed = getattr(results[0], 'speed', {}) if results else {}
            top_k_classes = utx.get_top_k_classes(results, model.names, k=top_classes_n)

            path_results_img = os.path.join(path_results, str(image_no))
            os.makedirs(path_results_img, exist_ok=True)

            detection_summary = {
                'image_id': str(image_no),
                'image_id_padded': image_id,
                'input_image_path': image_path,
                'sam_mask_path': str(sam_path),
                'refined_mask_path': str(refined_path),
                'fixed_category': fixed_category,
                'explained_class': fixed_category,
                'predicted_class_id': int(predicted_cls),
                'f_S': float(f_S),
                'f_0': float(f_0),
                'has_segmentation': bool(has_segmentation),
                'speed': to_jsonable(yolo_speed),
                'top_k_classes': [
                    {'class_id': int(class_id), 'class_name': str(class_name), 'confidence': float(confidence)}
                    for class_id, class_name, confidence in top_k_classes
                ],
            }
            detection_summary_path = os.path.join(path_results_img, f'{image_no}_results.json')
            with open(detection_summary_path, 'w', encoding='utf-8') as f:
                json.dump(to_jsonable(detection_summary), f, indent=2)

            masks_refined = np.load(refined_path).astype(np.uint16)
            if sorted_path.exists():
                masks_sorted = np.load(sorted_path).astype(np.uint16)
            else:
                masks_sorted = masks_refined
                if verbose:
                    print(f'Missing sorted partitions, using refined partitions for SAM_S: {sorted_path}')

            partitions_sorted = uts.sanitize_partitions(
                uts.cap_partition_labels(masks_sorted, max_labels=63),
                image_to_explain.shape,
                max_labels=63,
            )
            partitions_refined = uts.sanitize_partitions(
                uts.cap_partition_labels(masks_refined, max_labels=63),
                image_to_explain.shape,
                max_labels=63,
            )

            explainer = shap_bpt.Explainer(
                predict_yolo_masked,
                image_to_explain,
                num_explained_classes=num_explained_classes,
                verbose=verbose,
            )

            shap_values = {}
            auc_records = []

            method_steps = [
                ('BPT', None, {}),
                ('SAM_S', partitions_sorted, {'use_perim_term': False, 'use_area_term': False, 'use_color_term': False}),
                # ('SAM_R', partitions_refined, {'use_perim_term': False, 'use_area_term': False, 'use_color_term': False}),
                ('BPT_SAM_S', partitions_sorted, {'use_perim_term': True, 'use_area_term': True, 'use_color_term': True}),
                ('BPT_SAM_R', partitions_refined, {'use_perim_term': True, 'use_area_term': True, 'use_color_term': True}),
            ]

            for method_name, partitions, bpt_kwargs in tqdm(method_steps, desc=f'{image_id} methods', leave=False):
                if partitions is None:
                    shap_values_current = explainer.explain_instance(
                        MAX_EVALS_BUDGET,
                        method='BPT',
                        batch_size=eval_batch_size,
                    )
                else:
                    bptree = shap_bpt.build_bpt_from_image(
                        image_to_explain,
                        prebuilt_partitions=partitions,
                        **bpt_kwargs,
                    )
                    if verbose:
                        print(method_name, 'BPTree N:', bptree.N)
                    shap_values_current = explainer.explain_instance(
                        MAX_EVALS_BUDGET,
                        method='BPT',
                        bpt=bptree,
                        batch_size=eval_batch_size,
                    )

                shap_values[method_name] = shap_values_current
                figure_name = f'{method_name}_{image_no}.png'
                utx.plot_single_attributions(
                    shap_values_current,
                    path_results_img,
                    figure_name,
                    robust_percentile=99.9985,
                    save_plot=save_plots,
                    destroy_fig=destroy_figs,
                )

                auc_records.append({
                    'exp_code': method_name,
                    'label': method_name,
                    'description': experiment_map[method_name]['description'],
                    'shap_values': shap_values_current[0],
                })

            if compute_auc:
                auc_results = compute_auc_results(
                    auc_records,
                    predict_yolo_masked,
                    f_S,
                    f_0,
                    predicted_cls,
                    batch_size=auc_batch_size,
                    verbose=verbose,
                )
                plot_auc_results(
                    auc_results,
                    save_plot=save_plots,
                    image_no=image_no,
                    destroy_figs=destroy_figs,
                )
                data = auc_results_to_rows(
                    auc_results,
                    image_no=image_no,
                    image_id=image_id,
                    fixed_category=fixed_category,
                    f_S=f_S,
                    f_0=f_0,
                )
                auc_table_rows.extend(data)

            # report_path = utx.build_html_report(
            #     path_results_img,
            #     report_html,
            #     experiment_map,
            #     experiment_table,
            #     image_no,
            #     metadata_json=detection_summary_path,
            # )
            # if report_path is not None:
            #     completed_reports.append(str(report_path))
            #     print(f'Saved HTML report: {report_path}')

        except Exception as exc:
            failures.append({'image_id': image_id, 'error': repr(exc), 'traceback': traceback.format_exc()})
            print(f'FAILED {image_id}: {exc}')
            if verbose:
                traceback.print_exc()

    if completed_reports:
        all_report_path, all_report_df = utx.build_all_images_html_report(
            path_results,
            os.path.join(path_results, 'shapbpt_all_images_report.html'),
        )
        print(f'All-images report: {all_report_path}')

    if auc_table_rows:
        auc_all_df = pd.DataFrame(auc_table_rows)
        auc_all_path = os.path.join(path_results, 'auc_results_all_images_current_run.csv')
        auc_all_df.to_csv(auc_all_path, index=False)
        print(f'Current-run AUC table: {auc_all_path}')

    if failures:
        failures_path = os.path.join(path_results, 'full_image_set_failures.json')
        with open(failures_path, 'w', encoding='utf-8') as f:
            json.dump(failures, f, indent=2)
        print(f'Failures saved to: {failures_path}')

print(f'Completed reports: {len(completed_reports)}')
print(f'Failures: {len(failures)}')


In [ ]:
# Disabled in the full image-set test clone.
# Original cell started with: report_path = ut.build_html_report(path_results_img,
# Use the corrected full-pipeline cell below instead.


## END

In [ ]:
## Aggregate AUC table and box plots across all computed images
from pathlib import Path

path_results = Path(original_working_dir) / 'results'
current_run_path = path_results / f'auc_results_all_images_current_run_{path_partition_folder}_old_iou.csv'

if 'auc_table_rows' in globals() and len(auc_table_rows) > 0:
    auc_all = pd.DataFrame(auc_table_rows)
elif current_run_path.exists():
    auc_all = pd.read_csv(current_run_path)
else:
    raise FileNotFoundError(f'No combined AUC table found at {current_run_path}. Run the full-pipeline cell first.')

auc_all['method'] = pd.Categorical(
    auc_all['method'],
    categories=['BPT', 'SAM_S', 'SAM_R', 'BPT_SAM_S', 'BPT_SAM_R'],
    ordered=True,
)
auc_all = auc_all.sort_values(['method', 'image_no']).reset_index(drop=True)

summary = (
    auc_all
    .groupby('method', observed=True)
    .agg(
        images=('image_no', 'nunique'),
        auc_ins_mean=('auc_ins', 'mean'),
        auc_ins_std=('auc_ins', 'std'),
        auc_ins_median=('auc_ins', 'median'),
        auc_del_mean=('auc_del', 'mean'),
        auc_del_std=('auc_del', 'std'),
        auc_del_median=('auc_del', 'median'),
    )
    .reset_index()
)

display(auc_all)
display(summary)

auc_all.to_csv(path_results / 'auc_results_all_images.csv', index=False)
summary.to_csv(path_results / 'auc_results_summary.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
plot_data = [
    (axes[0], 'auc_ins', '$\mathit{AUC}^{+}$ across images', 'Higher is better'),
    (axes[1], 'auc_del', '$\mathit{AUC}^{-}$ across images', 'Lower is better'),
]

methods = [m for m in ['BPT', 'SAM_S', 'SAM_R', 'BPT_SAM_S', 'BPT_SAM_R'] if m in set(auc_all['method'].dropna().astype(str))]
for ax, metric, title, ylabel in plot_data:
    values = [auc_all.loc[auc_all['method'].astype(str) == method, metric].dropna().values for method in methods]
    ax.boxplot(values, labels=methods, showmeans=True, patch_artist=True)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=25)


plt.suptitle(f'Aggregate AUC Results Across {len(auc_all.image_no.unique())} Images & Budget: {MAX_EVALS_BUDGET}', fontsize=16)
plt.tight_layout()
boxplot_path = path_results / 'auc_results_boxplots.png'
plt.savefig(boxplot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved aggregate AUC table: {path_results / "auc_results_all_images.csv"}')
print(f'Saved aggregate summary: {path_results / "auc_results_summary.csv"}')
print(f'Saved box plot: {boxplot_path}')
